In [1]:
#!/usr/bin/env python
"""
LeafletFA Model Evaluation in Mouse 

This script:
1. Loads trained LeafletFA model outputs and associated data (mouse foundation smart-seq data)
2. Load the human foundation data and map junctions to human (via list of conserved junctions)
3. Apply the model to the human data
4. Save the predicted factor activities and factor usage
"""

import os
import sys
import glob
import pickle
import gzip
import warnings
from pathlib import Path
from collections import defaultdict
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder
from pyfaidx import Fasta

# Data analysis libraries
import numpy as np
import pandas as pd
import scipy
import scipy.stats as stats
import scipy.sparse as sp
from scipy.stats import spearmanr, pearsonr
from scipy.sparse import csr_matrix
from scipy.cluster.hierarchy import linkage, dendrogram
import scanpy as sc
import umap

# Machine learning libraries
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.metrics import (mean_squared_error, accuracy_score, r2_score, 
                           classification_report, confusion_matrix)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils import resample

# Statistical modeling
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from mord import OrdinalRidge

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import ScalarFormatter
from adjustText import adjust_text

# Single-cell analysis libraries
import anndata as ad
import scanpy as sc

# Bioinformatics libraries
import gffutils
from tqdm import tqdm

# PyTorch setup
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device count:", torch.cuda.device_count())
    print("CUDA device name:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

torch.set_default_tensor_type("torch.FloatTensor" if device.type == "cpu" else "torch.cuda.FloatTensor")
torch.manual_seed(0)

# Configure plotting and warnings
sns.set_theme()
sc.set_figure_params(figsize=(7, 7), frameon=True, dpi=80, facecolor='white')
warnings.filterwarnings('ignore')

# =============================================================================
# Custom Module Imports
# =============================================================================

# Add custom module paths
leaflet_src_path = "/gpfs/commons/home/kisaev/Leaflet-private/src/"
utils_path = "/gpfs/commons/home/kisaev/Leaflet-analysis/Multi_Species_Splicing_Foundation/shared_utils/"

for path in [leaflet_src_path, utils_path]:
    if path not in sys.path:
        sys.path.append(path)

# Import LeafletFA modules
import BetaDirichletFactor.LeafletFA as LeafletFA
import BetaDirichletFactor.utils as utilsFA

# Import utility functions
from utils import load_model
from utils import *
from figure_plotting import *
from atse_viz import *

Torch version: 2.4.1.post300
CUDA available: True
CUDA device count: 2
CUDA device name: Tesla V100-PCIE-16GB
Using device: cuda


/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/torch/__init__.py:955: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at /home/conda/feedstock_root/build_artifacts/libtorch_1728241823685/work/torch/csrc/tensor/python_tensor.cpp:432.)
  _C._set_default_tensor_type(t)


Torch Version: 2.4.1.post300
CUDA Version: 12.0
Added /gpfs/commons/home/kisaev/LeafletFA-utils to sys.path
Visualization imports successful!


In [2]:
BASE_DIR = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/MOUSE_SPLICING_FOUNDATION"
RESULTS_BASE_DIR = "/gpfs/commons/home/kisaev/Leaflet-analysis/Mouse_Splicing_Foundation/model_train/MOUSE_FOUNDATION/results"
MODEL_FILES = "/gpfs/commons/home/kisaev/Leaflet-analysis/Multi_Species_Splicing_Foundation/mouse_human_transfer_learning"

In [3]:
# Load atse_mapping, subset_splice_adata_mouse, subset_splice_adata_human from MODEL_FILES
atse_mapping = pd.read_csv(f"{MODEL_FILES}/atse_mapping.csv")
subset_splice_adata_mouse = ad.read_h5ad(f"{MODEL_FILES}/subset_splice_adata_mouse.h5ad")
subset_splice_adata_human = ad.read_h5ad(f"{MODEL_FILES}/subset_splice_adata_human.h5ad")
subset_splice_adata_human_with_mouse_transfer = ad.read_h5ad(f"{MODEL_FILES}/subset_splice_adata_human_with_mouse_transfer.h5ad")

In [5]:
subset_splice_adata_human_with_mouse_transfer.obs["dataset"].value_counts()

dataset
allen_brain       45496
tabula_sapiens    31490
Name: count, dtype: int64